# Week 2 — Data Cleaning & Preprocessing
**Dataset:** `annual_gold_rate.csv`

Tasks:
1. Handle missing values
2. Remove duplicates
3. Encode categorical variables
4. Normalize numerical features


## 1. Import Libraries & Load Data

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

df = pd.read_csv('annual_gold_rates.csv')
df.head()

,Date,Market,Purity,USD,EUR,GBP,INR,AED,CNY
0,1979-04,Kolkata,18K,231.76,140.77,108.84,1887.88,885.36,704.86
1,2002-09,Kolkata,22K,280.67,297.25,186.83,13633.03,1030.90,2323.10
2,2000-08,Delhi,24K,281.11,304.92,185.67,NaN,1032.50,2327.15
3,1990-11,Chennai,24K,383.12,282.62,215.71,6689.93,1406.18,1825.02
4,2002-06,New York,24K,303.26,321.17,201.86,14729.96,1113.85,2510.02


In [8]:
df.shape

(1000, 9)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    1000 non-null   str    
 1   Market  1000 non-null   str    
 2   Purity  1000 non-null   str    
 3   USD     969 non-null    float64
 4   EUR     977 non-null    float64
 5   GBP     974 non-null    float64
 6   INR     972 non-null    float64
 7   AED     969 non-null    float64
 8   CNY     972 non-null    float64
dtypes: float64(6), str(3)
memory usage: 70.4 KB


## 2. Explore Missing Values

In [10]:
df.isnull().sum()

Date       0
Market     0
Purity     0
USD       31
EUR       23
GBP       26
INR       28
AED       31
CNY       28
dtype: int64

In [11]:
# Percentage of missing values per column
(df.isnull().sum() / len(df) * 100).round(2)

Date      0.0
Market    0.0
Purity    0.0
USD       3.1
EUR       2.3
GBP       2.6
INR       2.8
AED       3.1
CNY       2.8
dtype: float64

## 3. Handle Missing Values

Numerical price columns (USD, EUR, GBP, INR, AED, CNY) are filled using the
**median** value grouped by `Purity`, since price naturally differs by purity
(24K vs 22K vs 18K). This is more accurate than a single global median.

In [12]:
num_cols = ['USD', 'EUR', 'GBP', 'INR', 'AED', 'CNY']

for col in num_cols:
    df[col] = df.groupby('Purity')[col].transform(lambda x: x.fillna(x.median()))

# Fallback in case any NaNs remain (e.g. whole group missing)
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

df.isnull().sum()

Date      0
Market    0
Purity    0
USD       0
EUR       0
GBP       0
INR       0
AED       0
CNY       0
dtype: int64

## 4. Remove Duplicates

In [13]:
print("Duplicate rows before:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows after:", df.duplicated().sum())
print("New shape:", df.shape)

Duplicate rows before: 36
Duplicate rows after: 0
New shape: (964, 9)


## 5. Encode Categorical Variables

`Market` and `Purity` are categorical (nominal) columns.
- `Purity` has a natural order (18K < 22K < 24K) → **Label Encoding**
- `Market` has no order → **One-Hot Encoding**

In [14]:
# Label encode Purity (ordinal)
purity_order = {'18K': 0, '22K': 1, '24K': 2}
df['Purity_encoded'] = df['Purity'].map(purity_order)

# One-hot encode Market (nominal)
df = pd.get_dummies(df, columns=['Market'], prefix='Market')

df.head()

,Date,Purity,USD,EUR,GBP,INR,AED,CNY,Purity_encoded,Market_Chennai,Market_Delhi,Market_Dubai,Market_Kolkata,Market_London,Market_Mumbai,Market_New York
0,1979-04,18K,231.76,140.77,108.84,1887.880,885.36,704.86,0,False,False,False,True,False,False,False
1,2002-09,22K,280.67,297.25,186.83,13633.030,1030.90,2323.10,1,False,False,False,True,False,False,False
2,2000-08,24K,281.11,304.92,185.67,13010.775,1032.50,2327.15,2,False,True,False,False,False,False,False
3,1990-11,24K,383.12,282.62,215.71,6689.930,1406.18,1825.02,2,True,False,False,False,False,False,False
4,2002-06,24K,303.26,321.17,201.86,14729.960,1113.85,2510.02,2,False,False,False,False,False,False,True


## 6. Normalize Numerical Features

In [15]:
scaler = MinMaxScaler()

df_normalized = df.copy()
df_normalized[num_cols] = scaler.fit_transform(df[num_cols])

df_normalized[num_cols].head()

,USD,EUR,GBP,INR,AED,CNY
0,0.054227,0.034161,0.027133,0.000559,0.024829,0.003363
1,0.083578,0.141954,0.087130,0.089653,0.049491,0.144196
2,0.083842,0.147237,0.086237,0.084933,0.049762,0.144549
3,0.145060,0.131876,0.109347,0.036986,0.113085,0.100849
4,0.097135,0.158431,0.098692,0.097974,0.063548,0.160464


## 7. Final Cleaned Dataset

In [16]:
df_normalized.drop(columns=['Purity']).head(10)

,Date,USD,EUR,GBP,INR,AED,CNY,Purity_encoded,Market_Chennai,Market_Delhi,Market_Dubai,Market_Kolkata,Market_London,Market_Mumbai,Market_New York
0,1979-04,0.054227,0.034161,0.027133,0.000559,0.024829,0.003363,0,False,False,False,True,False,False,False
1,2002-09,0.083578,0.141954,0.087130,0.089653,0.049491,0.144196,1,False,False,False,True,False,False,False
2,2000-08,0.083842,0.147237,0.086237,0.084933,0.049762,0.144549,2,False,True,False,False,False,False,False
3,1990-11,0.145060,0.131876,0.109347,0.036986,0.113085,0.100849,2,True,False,False,False,False,False,False
4,2002-06,0.097135,0.158431,0.098692,0.097974,0.063548,0.160464,2,False,False,False,False,False,False,True
5,2021-04,0.724324,0.722471,0.697477,0.742388,0.714070,0.698875,0,False,False,False,False,True,False,False
6,2017-05,0.473919,0.505590,0.499538,0.446173,0.454325,0.489575,0,False,False,False,True,False,False,False
7,2011-03,0.887004,0.739293,0.720902,0.560623,0.882752,0.852161,2,False,True,False,False,False,False,False
8,1996-09,0.151014,0.147217,0.137434,0.091711,0.119417,0.226421,2,False,False,False,True,False,False,False
9,1994-10,0.143650,0.155249,0.134649,0.076855,0.109353,0.227230,2,False,False,False,False,True,False,False


In [17]:
print("Final shape:", df_normalized.shape)
df_normalized.to_csv('cleaned_gold_rate.csv', index=False)
print("Saved as cleaned_gold_rate.csv")

Final shape: (964, 16)
Saved as cleaned_gold_rate.csv
